# GenRec: Card2Vec Attention-VAE for Commander recommendations

## 0 — Research question

Given a Commander and/or partial Commander deck, can we recover plausible missing cards? This held-out-card task learns from real deck construction and avoids subjective power labels.

The hierarchy is: **card-level co-occurrence → 896-d Card2Vec; partial unordered deck → set attention; deck-level structure → variational or deterministic latent; denoising objective → hidden-card scores**. EDHREC is used only as an external qualitative benchmark.

## 1 — Imports and configuration
All experimental choices live in this cell. GPU floating-point execution may still have small nondeterministic differences.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import copy, gc, hashlib, heapq, os, random, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from mtgdeck.data import *
from mtgdeck.card2vec import *
from mtgdeck.legality import *
from mtgdeck.recommend import *
from mtgdeck.vae import *

CLEAN_DATA = ROOT/'data/decks_clean.jsonl'
from mtgdeck.metadata import default_oracle_path
ORACLE_DATA=default_oracle_path(ROOT/'data')
CARD2VEC_DIM=896
IDENTITY_SCHEMA='oracleid_v2'  # forces fresh artifacts after replacing ambiguous name tokens
VARIATIONAL=True  # primary run
FREEZE_CARD2VEC=True  # primary run
RUN_ABLATIONS=True  # sequentially run deterministic/frozen and variational/fine-tuned after the primary run
variant='variational' if VARIATIONAL else 'deterministic'
embedding_variant='frozen' if FREEZE_CARD2VEC else 'finetuned'
CFG = dict(data=CLEAN_DATA, oracle=ORACLE_DATA, commander_eligibility=ROOT/'data/commander_eligible_oracle_ids.json', card2vec=ROOT/f'data/card2vec_clean_{IDENTITY_SCHEMA}_{CARD2VEC_DIM}.model', checkpoint=ROOT/f'checkpoints/attention_{IDENTITY_SCHEMA}_{variant}_{embedding_variant}_{CARD2VEC_DIM}.pt', seed=42, device='cuda' if torch.cuda.is_available() else 'cpu', n_jobs=min(6, max(1, (os.cpu_count() or 2)-1)), batch_size=32, eval_batch_size=128, card2vec_dim=CARD2VEC_DIM, freeze_card2vec=FREEZE_CARD2VEC, variational=VARIATIONAL, model_dim=256, latent_dim=64, heads=8, blocks=2, pool_queries=4, decoder_queries=4, initial_logit_scale=10.0, primary_mask_ratio=0.20, train_mask_ratios=(0.10,0.20,0.30,0.40,0.50), validation_mask_ratios=(0.10,0.20,0.30,0.40,0.50), test_mask_repeats=3, learning_rate=2e-4, epochs=8, kl_beta=0.01, kl_warmup_steps=5000, prior_modes=('log_count','conditional','pmi'), prior_smoothing=0.5, hybrid_weights=(0.0,0.1,0.25,0.5,1.0,1.5,2.0), near_duplicate_threshold=0.90, near_duplicate_minhash_components=128, near_duplicate_band_size=8, near_duplicate_sample_per_split=1000, evaluation_k=(10,20,50), run_ablations=RUN_ABLATIONS, ablations=(('deterministic_frozen',False,True),('variational_finetuned',True,False)))
random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
device = torch.device(CFG['device']); CFG


## 2 — Dataset audit and canonical model identities
`Run All` reads canonical local data only; collection is deliberately separate (`python -m scrape.scraper`). Human-readable names are retained for presentation, while every model-facing card token is the stable canonical Oracle ID.


In [ ]:
if not CFG['data'].exists(): raise FileNotFoundError('Create and curate data/decks_clean.jsonl first')
raw_rows = list(iter_jsonl(CFG['data']))
errors = [(i, canonical_validation_errors(row)) for i, row in enumerate(raw_rows) if canonical_validation_errors(row)]
decks = [row for row in raw_rows if not canonical_validation_errors(row)]
catalog = OracleCatalog.from_path(CFG['oracle'], CFG['commander_eligibility'])

audit = pd.DataFrame({
 'source': [d['source'] for d in decks], 'date': [d.get('date') for d in decks],
 'cards': [sum(x['quantity'] for z in ('commanders','mainboard') for x in d[z]) for d in decks],
 'unique_cards': [len(deck_card_names(d)) for d in decks],
 'commander_count': [len(d['commanders']) for d in decks]})
all_quantities = [x['quantity'] for d in decks for z in ('commanders','mainboard','sideboard') for x in d[z]]
summary = {'decks': len(decks), 'sources': audit.source.value_counts().to_dict(), 'unique_cards': len(set().union(*(deck_card_names(d) for d in decks))) if decks else 0, 'unique_commanders': len({commander_key(d) for d in decks}), 'date_range': (audit.date.dropna().min(), audit.date.dropna().max()), 'exact_duplicates': len(decks)-len({deck_fingerprint(d) for d in decks}), 'missing_commander': int((audit.commander_count==0).sum()), 'malformed': len(errors), 'quantity_p50_p99': np.percentile(all_quantities, [50,99]).tolist() if all_quantities else []}
display(summary); display(audit.describe(include='all'))
audit.cards.hist(bins=30); plt.title('Physical cards per deck'); plt.show()

card_display_by_token={}
def oracle_token(item):
    name=str(item.get('name','')) if isinstance(item,dict) else str(item)
    oracle_id=item.get('oracle_id') if isinstance(item,dict) else None
    card=catalog.resolve(name,str(oracle_id) if oracle_id else None)
    if card is None: raise ValueError(f'card was not found in Oracle catalog: {name}')
    token=ORACLE_TOKEN_PREFIX+str(card['oracle_id']).casefold()
    card_display_by_token[token]=str(card['name'])
    return token,str(card['oracle_id'])
def modelize_deck(record,in_place=False):
    result=record if in_place else copy.deepcopy(record)
    for zone in ('commanders','mainboard','sideboard'):
        for item in result.get(zone,[]):
            token,oracle_id=oracle_token(item); item['name']=token; item['oracle_id']=oracle_id
    return result
def display_card(token): return card_display_by_token.get(normalize_card_name(token),str(token))
for deck in decks: modelize_deck(deck,in_place=True)
assert all(str(item['name']).startswith(ORACLE_TOKEN_PREFIX) for deck in decks for zone in ('commanders','mainboard') for item in deck[zone])
print({'identity_schema':CFG.get('card2vec').stem,'oracle_identity_tokens':len(card_display_by_token),'card2vec_rebuild_path':CFG['card2vec'],'checkpoint_rebuild_path':CFG['checkpoint']})
del raw_rows, errors


## 3 — Near-duplicate-grouped splits and partial-deck masking
Exact and near-duplicate decks are kept in one connected component before deterministic splitting. Candidate pairs come from deterministic MinHash bands and are accepted only after exact Jaccard verification; 128 hashes in bands of eight makes misses at the 0.90 threshold very unlikely without an all-pairs comparison. We hide unique mainboard identities while retaining every commander.


In [ ]:
def grouped_near_duplicate_split(rows,seed=42,ratios=(0.8,0.1,0.1)):
    components=int(CFG['near_duplicate_minhash_components']); band_size=int(CFG['near_duplicate_band_size'])
    if components%band_size: raise ValueError('near_duplicate_minhash_components must be divisible by band size')
    threshold=float(CFG['near_duplicate_threshold']); hash_cache={}; buckets=defaultdict(list)
    parent=list(range(len(rows))); rank=[0]*len(rows); sizes=[len(deck_card_names(deck)) for deck in rows]
    def find(index):
        while parent[index]!=index:
            parent[index]=parent[parent[index]]; index=parent[index]
        return index
    def union(left,right):
        left,right=find(left),find(right)
        if left==right: return False
        if rank[left]<rank[right]: left,right=right,left
        parent[right]=left
        if rank[left]==rank[right]: rank[left]+=1
        return True
    def hashes(card):
        if card not in hash_cache:
            encoded=card.encode('utf-8')
            hash_cache[card]=np.fromiter((int.from_bytes(hashlib.blake2b(i.to_bytes(2,'little')+encoded,digest_size=8).digest(),'little') for i in range(components)),dtype=np.uint64,count=components)
        return hash_cache[card]
    candidate_comparisons=near_edges=0
    for index,deck in enumerate(rows):
        cards=deck_card_names(deck)
        signature=np.stack([hashes(card) for card in cards]).min(axis=0)
        keys=[tuple(int(value) for value in signature[start:start+band_size]) for start in range(0,components,band_size)]
        candidates=set()
        for key in keys: candidates.update(buckets.get(key,()))
        for other in candidates:
            if min(sizes[index],sizes[other])/max(sizes[index],sizes[other])<threshold: continue
            candidate_comparisons+=1; other_cards=deck_card_names(rows[other])
            similarity=len(cards&other_cards)/len(cards|other_cards)
            if similarity>=threshold:
                near_edges+=1; union(index,other)
        for key in keys: buckets[key].append(index)
    grouped=defaultdict(list)
    for index,deck in enumerate(rows): grouped[find(index)].append(deck)
    groups=list(grouped.values()); random.Random(seed).shuffle(groups)
    partitions=[[],[],[]]; capacities=[len(rows)*ratio for ratio in ratios]
    for group in groups:
        destination=min(range(3),key=lambda i:(len(partitions[i])/capacities[i],i))
        partitions[destination].extend(group)
    report={'groups':len(groups),'decks_joined_to_existing_groups':len(rows)-len(groups),'verified_near_edges':near_edges,'candidate_comparisons':candidate_comparisons,'minhash_components':components,'band_size':band_size,'threshold':threshold}
    return (*partitions,report)

train_decks,val_decks,test_decks,near_grouping=grouped_near_duplicate_split(decks,CFG['seed'])
split_rows={'train':train_decks,'validation':val_decks,'test':test_decks}
split_fingerprints={name:{deck_fingerprint(deck) for deck in rows} for name,rows in split_rows.items()}
exact_cross_split={f'{left}-{right}':len(split_fingerprints[left]&split_fingerprints[right]) for left,right in (('train','validation'),('train','test'),('validation','test'))}
assert not any(exact_cross_split.values()),exact_cross_split
def sampled_near_duplicates(left_rows,right_rows,left_seed,right_seed):
    left=random.Random(left_seed).sample(left_rows,min(len(left_rows),CFG['near_duplicate_sample_per_split']))
    right=random.Random(right_seed).sample(right_rows,min(len(right_rows),CFG['near_duplicate_sample_per_split']))
    right_groups=defaultdict(list)
    for deck in right: right_groups[len(deck_card_names(deck))].append((deck['deck_id'],deck_card_names(deck)))
    match_count=0; examples=[]; threshold=CFG['near_duplicate_threshold']
    for deck in left:
        cards=deck_card_names(deck); low=int(np.ceil(len(cards)*threshold)); high=int(np.floor(len(cards)/threshold))
        for size in range(low,high+1):
            for other_id,other_cards in right_groups.get(size,()):
                similarity=len(cards&other_cards)/len(cards|other_cards)
                if similarity>=threshold:
                    match_count+=1
                    if len(examples)<5: examples.append((deck['deck_id'],other_id,similarity))
    return {'sampled_left':len(left),'sampled_right':len(right),'near_pairs':match_count,'examples':examples}
near_cross_split={f'{left}-{right}':sampled_near_duplicates(split_rows[left],split_rows[right],CFG['seed']+i*2,CFG['seed']+i*2+1) for i,(left,right) in enumerate((('train','validation'),('train','test'),('validation','test')))}
assert not any(result['near_pairs'] for result in near_cross_split.values()),near_cross_split
display({'split_sizes':{name:len(rows) for name,rows in split_rows.items()},'near_duplicate_grouping':near_grouping,'exact_cross_split':exact_cross_split,'sampled_near_duplicate_audit':near_cross_split})
example_visible,example_hidden=mask_deck(train_decks[0],CFG['primary_mask_ratio'],seed=CFG['seed'])
print(f"Original: {sum(x['quantity'] for z in ('commanders','mainboard') for x in train_decks[0][z])} cards; visible tokens: {len(example_visible['commanders'])+len(example_visible['mainboard'])}; hidden tokens: {len(example_hidden)}")
display(pd.DataFrame([{**item,'name':display_card(item['name'])} for item in example_hidden]).head())


## 4 — Simple baselines
Global popularity, exact commander-conditioned popularity, and observed co-occurrence share the same three deterministic test masks and Oracle-ID legality filtering as the learned model.


In [ ]:
vocab=build_vocabulary(train_decks,min_count=2)
legal_candidates=CommanderCandidateIndex(catalog,vocab)
legal_mask_cache={}
def legal_mask_for(deck):
    key=commander_key(deck)
    if key not in legal_mask_cache: legal_mask_cache[key]=legal_candidates.allowed_mask(deck)
    return legal_mask_cache[key]
train_commander_keys={commander_key(deck) for deck in train_decks}
baseline_index=fit_baselines(train_decks,n_jobs=CFG['n_jobs'],parallel_verbose=10)
commander_deck_counts=Counter(commander_key(deck) for deck in train_decks)
assert sum(commander_deck_counts.values())==len(train_decks)
print({'baseline_training_decks':len(train_decks),'baseline_commanders':len(commander_deck_counts),'test_masks_per_deck':CFG['test_mask_repeats']})
def evaluation_mask_seed(index,ratio,repeat=0): return CFG['seed']+index+repeat*1_000_003+int(round(ratio*100))*10_000_019
def held_out_examples(rows,ratio=CFG['primary_mask_ratio'],repeats=1):
    for repeat in range(repeats):
        for i,complete in enumerate(rows):
            visible,hidden=mask_deck(complete,ratio,seed=evaluation_mask_seed(i,ratio,repeat))
            yield visible,[normalize_card_name(x['name']) for x in hidden]
def evaluate_ranker(rank_fn,rows=test_decks,ratio=CFG['primary_mask_ratio'],repeats=1):
    metric_rows=[]
    for visible,hidden in held_out_examples(rows,ratio,repeats):
        ranked=rank_fn(visible)
        metric_rows.append({**{f'Recall@{k}':recall_at_k(ranked,hidden,k) for k in CFG['evaluation_k']},**{f'NDCG@{k}':ndcg_at_k(ranked,hidden,k) for k in (10,20)}})
    return pd.DataFrame(metric_rows).mean().to_dict()
global_ranking=rank_scores(global_popularity_scores(baseline_index),k=len(baseline_index.vocabulary))
commander_rankings={key:rank_scores(counts,k=len(counts)) for key,counts in baseline_index.commander_counts.items()}
def take_unseen(ranking,visible,commanders,allowed,k=max(CFG['evaluation_k'])):
    present={normalize_card_name(name) for name in [*visible,*commanders]}
    return [(name,score) for name,score in ranking if name not in present and name in vocab and allowed[vocab[name]]][:k]
def rank_scores_legal(scores,present,allowed,k=max(CFG['evaluation_k'])):
    ranked=[]
    for raw_name,score in scores.items():
        name=normalize_card_name(raw_name); index=vocab.get(name)
        if name not in present and index is not None and allowed[index] and np.isfinite(score): ranked.append((name,float(score)))
    return heapq.nsmallest(k,ranked,key=lambda item:(-item[1],item[0]))
def cached_baseline_ranker(deck,method):
    visible=[x['name'] for x in deck['mainboard']]; commanders=[x['name'] for x in deck['commanders']]; allowed=legal_mask_for(deck)
    if method=='global': return take_unseen(global_ranking,visible,commanders,allowed)
    if method=='commander': return take_unseen(commander_rankings.get(commander_key(commanders),()),visible,commanders,allowed)
    present={normalize_card_name(name) for name in [*visible,*commanders]}
    return rank_scores_legal(cooccurrence_scores(baseline_index,[*visible,*commanders]),present,allowed)
baseline_results={}
for method in ('global','commander','cooccurrence'):
    baseline_results[method]=evaluate_ranker(lambda d,m=method:cached_baseline_ranker(d,m),repeats=CFG['test_mask_repeats'])
pd.DataFrame(baseline_results).T


## 5 — Card2Vec
Each deck is an unordered sentence of unique canonical Oracle-ID tokens. Quantity remains a separate model feature. The identity-schema suffix deliberately forces a new 896-dimensional Card2Vec artifact on the next run.


In [ ]:
corpus=build_card2vec_corpus(train_decks)
if CFG['card2vec'].exists(): c2v=load_card2vec(CFG['card2vec'])
else:
    c2v=train_card2vec(corpus,vector_size=CFG['card2vec_dim'],window=100,min_count=2,epochs=10,workers=CFG['n_jobs'],seed=CFG['seed'])
    save_card2vec(c2v,CFG['card2vec'])
if c2v.wv.vector_size!=CFG['card2vec_dim']: raise ValueError(f"Card2Vec dimension {c2v.wv.vector_size} does not match configured {CFG['card2vec_dim']}")
if set(c2v.wv.key_to_index)&{name for name in vocab if not name.startswith(ORACLE_TOKEN_PREFIX) and name not in ('<PAD>','<UNK>')}: raise AssertionError('Card2Vec contains legacy name tokens')
def display_neighbors(name,k=5):
    token,_=oracle_token(name)
    return [(display_card(other),score) for other,score in c2v.wv.most_similar(token,topn=k)]
for name in ('Ponder','Demonic Tutor','Sol Ring','Rhystic Study'): print(name,display_neighbors(name,5))
def legal_card2vec_ranker(deck):
    visible=[x['name'] for x in deck['mainboard']]; commanders=[x['name'] for x in deck['commanders']]; present={normalize_card_name(name) for name in [*visible,*commanders]}
    return rank_scores_legal(card2vec_scores(c2v,[*visible,*commanders]),present,legal_mask_for(deck))
card2vec_result=evaluate_ranker(legal_card2vec_ranker,repeats=CFG['test_mask_repeats'])
card2vec_result


## 6 — Variational denoising set-completion architecture
For token matrix $X$, self-attention constructs $Q=XW_Q$, $K=XW_K$, and $V=XW_V$, then mixes cards by $\mathrm{softmax}(QK^T/\sqrt{d})V$. There is no positional encoding: reordering a deck must only reorder intermediate card states. Learned cross-attention queries pool several deck facets.

This is a **denoising set-completion model**: partial deck $\rightarrow$ position-free set encoder $\rightarrow$ latent $z$ $\rightarrow$ hidden-card scores. In variational mode the encoder parameterizes $q(z|D)=\mathcal{N}(\mu,\mathrm{diag}(\sigma^2))$ and samples $z=\mu+\sigma\epsilon$; deterministic mode uses $z=\mu$ and disables KL. Decoder queries score the vocabulary directly in frozen Card2Vec space through a learnable temperature. That shared geometry is efficient but constraining; `freeze_card2vec=False` is the direct fine-tuning ablation. Commander legality masks remove banned and off-color candidates during both training and inference.

In [ ]:
weights=torch.from_numpy(embedding_matrix(c2v,vocab))
assert weights.shape==(len(vocab),CFG['card2vec_dim']),weights.shape
def build_model(variational,freeze_card2vec):
    candidate=Card2VecAttentionVAE(weights.clone(),model_dim=CFG['model_dim'],num_heads=CFG['heads'],num_layers=CFG['blocks'],latent_dim=CFG['latent_dim'],num_pool_queries=CFG['pool_queries'],num_decoder_queries=CFG['decoder_queries'],freeze_card2vec=freeze_card2vec,initial_logit_scale=CFG['initial_logit_scale'],variational=variational).to(device)
    assert candidate.card_embedding.weight.requires_grad is not freeze_card2vec
    return candidate
model=build_model(CFG['variational'],CFG['freeze_card2vec'])
print({'vocab':len(vocab),'card2vec_dim':model.card_dim,'identity_schema':IDENTITY_SCHEMA,'frozen_card2vec':CFG['freeze_card2vec'],'variational':model.variational,'oracle_cards':len(catalog),'train_commanders':len(train_commander_keys),'initial_logit_scale':model.logit_scale.exp().item()})
model


## 7 — Unit and smoke checks

In [ ]:
def assert_targets_recommendable(targets,allowed,ids,padding,context):
    positive=targets.bool(); present=torch.zeros_like(positive)
    for row in range(ids.shape[0]): present[row,ids[row,~padding[row]].unique()]=True
    illegal=(positive&~allowed).nonzero(as_tuple=False); already_present=(positive&present).nonzero(as_tuple=False)
    if len(illegal) or len(already_present):
        def examples(positions): return [(int(row),display_card(inverse_vocab[int(column)])) for row,column in positions[:10].detach().cpu()]
        raise AssertionError(f'{context}: masked positive targets; illegal={examples(illegal)}, already_present={examples(already_present)}')

inverse_vocab={index:name for name,index in vocab.items()}
corpus_target_violations=[]
for deck in decks:
    allowed=legal_mask_for(deck); commander_tokens={normalize_card_name(x['name']) for x in deck['commanders']}; main_tokens=[]
    for item in deck['mainboard']:
        token=normalize_card_name(item['name']); main_tokens.append(token); index=vocab.get(token)
        if index is not None and not allowed[index]: corpus_target_violations.append((deck['deck_id'],display_card(token)))
    overlap=commander_tokens&set(main_tokens)
    if overlap: corpus_target_violations.extend((deck['deck_id'],display_card(token)) for token in overlap)
assert not corpus_target_violations,corpus_target_violations[:20]
print({'corpus_positive_target_violations':0,'decks_checked':len(decks)})

smoke_visible_hidden=[mask_deck(d,CFG['primary_mask_ratio'],evaluation_mask_seed(i,CFG['primary_mask_ratio'])) for i,d in enumerate(train_decks[:4])]
smoke_visible=[visible for visible,_hidden in smoke_visible_hidden]
rows=[deck_to_tokens(visible,vocab) for visible in smoke_visible]
ids,roles,qty,padding=[x.to(device) for x in collate_token_rows(rows,vocab['<PAD>'])]
out=model(ids,roles,qty,padding)
allowed=torch.as_tensor(np.stack([legal_mask_for(d) for d in smoke_visible]),device=device)
targets=torch.zeros(len(rows),len(vocab),device=device)
for i,(_visible,hidden) in enumerate(smoke_visible_hidden):
    for x in hidden:
        key=normalize_card_name(x['name'])
        if key in vocab: targets[i,vocab[key]]=1
assert_targets_recommendable(targets,allowed,ids,padding,'smoke check')
out['logits']=mask_present_logits(out['logits'],ids,padding).masked_fill(~allowed,-torch.inf)
assert torch.isfinite(out['logits'][targets.bool()]).all()
losses=vae_loss(out,targets,beta=0.0,include_kl=CFG['variational'])
assert all(torch.isfinite(value) for value in losses),losses
print('batch/mask',ids.shape,padding.shape,'mu/logvar/z',out['mu'].shape,out['logvar'].shape,out['latent'].shape,'queries/logits',out['decoder_queries'].shape,out['logits'].shape,'finite loss',True)

prior_smoothing=float(CFG['prior_smoothing']); prior_vocab_size=max(1,len(vocab)-2)
global_log_probability=np.zeros(len(vocab),dtype=np.float32); global_denominator=len(train_decks)+prior_smoothing*prior_vocab_size
for name,index in vocab.items():
    if name not in ('<PAD>','<UNK>'): global_log_probability[index]=np.log((baseline_index.global_counts.get(name,0)+prior_smoothing)/global_denominator)
commander_prior_cache={}
def commander_prior_vector(deck,mode='log_count'):
    commander=commander_key(deck); cache_key=(mode,commander)
    if cache_key in commander_prior_cache: return commander_prior_cache[cache_key]
    vector=np.zeros(len(vocab),dtype=np.float32); commander_decks=commander_deck_counts.get(commander,0)
    if mode!='none' and commander_decks:
        counts=baseline_index.commander_counts.get(commander,{})
        if mode=='log_count':
            for name,count in counts.items():
                if name in vocab: vector[vocab[name]]=np.log1p(count)
        elif mode in ('conditional','pmi'):
            denominator=commander_decks+prior_smoothing*prior_vocab_size; vector.fill(np.log(prior_smoothing/denominator))
            for name,count in counts.items():
                if name in vocab: vector[vocab[name]]=np.log((count+prior_smoothing)/denominator)
            if mode=='pmi': vector-=global_log_probability
        else: raise ValueError(f'unknown commander prior mode: {mode}')
    commander_prior_cache[cache_key]=vector
    return vector
def metric_row(ranked,hidden): return {**{f'Recall@{k}':recall_at_k(ranked,hidden,k) for k in CFG['evaluation_k']},**{f'NDCG@{k}':ndcg_at_k(ranked,hidden,k) for k in (10,20)}}
def evaluate_vae_batched(rows,prior_configs=(('none',0.0),),mask_ratio=CFG['primary_mask_ratio'],mask_repeats=1,include_groups=False,candidate_model=None):
    active_model=model if candidate_model is None else candidate_model
    configs=tuple(dict.fromkeys((str(mode),float(weight)) for mode,weight in prior_configs)); totals={}
    for config in configs: totals[config]={group:{'tasks':0,**{key:0.0 for key in ('Recall@10','Recall@20','Recall@50','NDCG@10','NDCG@20')}} for group in ('overall','seen','unseen')}
    prior_modes={mode for mode,weight in configs if mode!='none' and weight!=0}
    for repeat in range(mask_repeats):
        for start in range(0,len(rows),CFG['eval_batch_size']):
            complete_batch=rows[start:start+CFG['eval_batch_size']]; visible_batch=[]; hidden_batch=[]
            for offset,complete in enumerate(complete_batch):
                visible,hidden=mask_deck(complete,mask_ratio,seed=evaluation_mask_seed(start+offset,mask_ratio,repeat))
                visible_batch.append(visible); hidden_batch.append([normalize_card_name(x['name']) for x in hidden])
            allowed=np.stack([legal_mask_for(deck) for deck in visible_batch]); base_scores=score_cards_batch(active_model,visible_batch,vocab,device,allowed)
            prior_batches={mode:torch.as_tensor(np.stack([commander_prior_vector(deck,mode) for deck in visible_batch]),device=device) for mode in prior_modes}
            for config in configs:
                mode,weight=config; scores=base_scores if mode=='none' or weight==0 else base_scores+weight*prior_batches[mode]
                indices=torch.topk(scores,max(CFG['evaluation_k']),dim=1).indices.cpu().tolist()
                for complete,ranked_ids,hidden in zip(complete_batch,indices,hidden_batch):
                    ranked=[inverse_vocab[index] for index in ranked_ids]; metrics=metric_row(ranked,hidden); groups=['overall']
                    if include_groups: groups.append('seen' if commander_key(complete) in train_commander_keys else 'unseen')
                    for group in groups:
                        totals[config][group]['tasks']+=1
                        for key,value in metrics.items(): totals[config][group][key]+=value
    for config in configs:
        for group,values in totals[config].items():
            count=values['tasks']
            if count: totals[config][group]={key:(value/count if key!='tasks' else value) for key,value in values.items()}
    return totals


## 8 — Training
Each deck receives one freshly seeded mask per epoch, sampled from `train_mask_ratios`. Every batch asserts that all positive targets remain legal, absent from the visible deck, and finite after masking before backpropagation. Reconstruction, KL, beta, and total loss are logged separately; any non-finite value aborts immediately. Validation Recall@20 selects each experiment checkpoint.


In [ ]:
no_prior=('none',0.0)
def train_experiment(experiment_model,checkpoint,variational,label,epochs=None):
    epochs=CFG['epochs'] if epochs is None else int(epochs)
    optimizer=torch.optim.AdamW((p for p in experiment_model.parameters() if p.requires_grad),lr=CFG['learning_rate'])
    history=[]; step=0; best_val_recall20=-float('inf'); checkpoint.parent.mkdir(parents=True,exist_ok=True)
    steps_per_epoch=(len(train_decks)+CFG['batch_size']-1)//CFG['batch_size']
    if variational:
        assert kl_beta(0,CFG['kl_warmup_steps'],CFG['kl_beta'])==0
        assert kl_beta(CFG['kl_warmup_steps'],CFG['kl_warmup_steps'],CFG['kl_beta'])==CFG['kl_beta']
    print({'experiment':label,'steps_per_epoch':steps_per_epoch,'kl_beta_max':CFG['kl_beta'] if variational else 0.0,'kl_warmup_steps':CFG['kl_warmup_steps'],'warmup_epochs':CFG['kl_warmup_steps']/steps_per_epoch if variational else 0.0,'train_mask_ratios':CFG['train_mask_ratios'],'checkpoint':checkpoint})
    for epoch in range(epochs):
        experiment_model.train(); shuffled=list(train_decks); random.Random(CFG['seed']+epoch).shuffle(shuffled)
        epoch_losses=[]; epoch_mask_counts=Counter()
        for start in range(0,len(shuffled),CFG['batch_size']):
            batch=shuffled[start:start+CFG['batch_size']]; visible_hidden=[]
            for i,deck in enumerate(batch):
                mask_seed=CFG['seed']+epoch*len(shuffled)+start+i; ratio=random.Random(mask_seed+97).choice(CFG['train_mask_ratios']); epoch_mask_counts[ratio]+=1
                visible,hidden=mask_deck(deck,ratio,mask_seed); visible_hidden.append((visible,hidden,ratio))
            token_rows=[deck_to_tokens(visible,vocab) for visible,_hidden,_ratio in visible_hidden]
            ids,roles,qty,padding=[x.to(device) for x in collate_token_rows(token_rows,vocab['<PAD>'])]
            targets=torch.zeros(len(token_rows),len(vocab),device=device)
            for i,(_visible,hidden,_ratio) in enumerate(visible_hidden):
                for x in hidden:
                    key=normalize_card_name(x['name'])
                    if key in vocab: targets[i,vocab[key]]=1
            if not targets.sum().item(): continue
            allowed=torch.as_tensor(np.stack([legal_mask_for(visible) for visible,_hidden,_ratio in visible_hidden]),device=device)
            assert_targets_recommendable(targets,allowed,ids,padding,f'{label} epoch={epoch+1} batch_start={start}')
            beta=kl_beta(step,CFG['kl_warmup_steps'],CFG['kl_beta']) if variational else 0.0
            outputs=experiment_model(ids,roles,qty,padding); outputs['logits']=mask_present_logits(outputs['logits'],ids,padding).masked_fill(~allowed,-torch.inf)
            assert torch.isfinite(outputs['logits'][targets.bool()]).all(),f'{label}: non-finite positive logits before loss'
            total,rec,kl=vae_loss(outputs,targets,beta,include_kl=variational)
            assert all(torch.isfinite(value) for value in (total,rec,kl)),f'{label}: non-finite loss at epoch={epoch+1}, batch_start={start}'
            optimizer.zero_grad(); total.backward(); optimizer.step(); step+=1; epoch_losses.append((total.item(),rec.item(),kl.item(),beta))
        means=np.mean(epoch_losses,axis=0); val_metrics=evaluate_vae_batched(val_decks,(no_prior,),CFG['primary_mask_ratio'],candidate_model=experiment_model)[no_prior]['overall']
        row={'experiment':label,'epoch':epoch+1,'total_loss':means[0],'reconstruction_loss':means[1],'kl_loss':means[2],'beta_mean':means[3],'beta_end':beta,'logit_scale':experiment_model.logit_scale.exp().item(),'mask_counts':dict(sorted(epoch_mask_counts.items())),**{f'val_{key}':value for key,value in val_metrics.items() if key!='tasks'}}; history.append(row); print(row)
        if val_metrics.get('Recall@20',0)>best_val_recall20:
            best_val_recall20=val_metrics['Recall@20']; serializable_cfg={key:(str(value) if isinstance(value,Path) else value) for key,value in CFG.items()}
            torch.save({'state_dict':experiment_model.state_dict(),'vocab':vocab,'config':{**serializable_cfg,'experiment':label,'variational':variational},'epoch':epoch+1,'val_metrics':val_metrics},checkpoint)
    best=torch.load(checkpoint,map_location=device,weights_only=False); experiment_model.load_state_dict(best['state_dict']); print('Loaded best',label,'epoch',best['epoch'],best['val_metrics'])
    return experiment_model,history,best

model,history,best=train_experiment(model,CFG['checkpoint'],CFG['variational'],f'{variant}_{embedding_variant}')
hybrid_configs=(no_prior,)+tuple((mode,float(weight)) for mode in CFG['prior_modes'] for weight in CFG['hybrid_weights'] if weight>0)
hybrid_validation=evaluate_vae_batched(val_decks,hybrid_configs,CFG['primary_mask_ratio'])
hybrid_validation_table=pd.DataFrame({f'{mode}:{weight:g}':groups['overall'] for (mode,weight),groups in hybrid_validation.items()}).T
best_hybrid_config=max(hybrid_configs,key=lambda config:(hybrid_validation[config]['overall']['Recall@20'],hybrid_validation[config]['overall']['NDCG@20']))
print('Selected commander prior',best_hybrid_config); display(hybrid_validation_table.sort_values(['Recall@20','NDCG@20'],ascending=False))
validation_by_mask=[]
for ratio in CFG['validation_mask_ratios']:
    configs=tuple(dict.fromkeys((no_prior,best_hybrid_config))); results=hybrid_validation if ratio==CFG['primary_mask_ratio'] else evaluate_vae_batched(val_decks,configs,ratio)
    for config in configs: validation_by_mask.append({'mask_ratio':ratio,'model':f'{config[0]}:{config[1]:g}',**results[config]['overall']})
display(pd.DataFrame(validation_by_mask).set_index(['mask_ratio','model'])[['tasks','Recall@10','Recall@20','Recall@50','NDCG@10','NDCG@20']])
pd.DataFrame(history).set_index('epoch')[['total_loss','reconstruction_loss','kl_loss','beta_mean','logit_scale','val_Recall@20','val_NDCG@20']].plot(subplots=True,figsize=(8,13)); plt.show()


## 9 — Matched-mask comparison and configured ablations
All test methods use the same three deterministic masks per deck. The primary checkpoint and commander prior are selected on validation data. When `run_ablations=True`, the notebook subsequently trains the deterministic/frozen and variational/fine-tuned variants with the identical architecture, masks, and checkpoint rule, then reports their validation and test metrics without using them to alter the primary result.


In [ ]:
test_configs=tuple(dict.fromkeys((no_prior,best_hybrid_config)))
test_evaluation=evaluate_vae_batched(test_decks,test_configs,CFG['primary_mask_ratio'],CFG['test_mask_repeats'],include_groups=True)
vae_result=test_evaluation[no_prior]['overall']; hybrid_result=test_evaluation[best_hybrid_config]['overall']
model_label='Variational' if CFG['variational'] else 'Deterministic'
comparison={**baseline_results,'Card2Vec':card2vec_result,f'Legal {model_label} Set Completion':vae_result,f'Legal {model_label} + {best_hybrid_config[0]}':hybrid_result}
print({'checkpoint_selection':'validation Recall@20','primary_mask_ratio':CFG['primary_mask_ratio'],'masks_per_test_deck_all_methods':CFG['test_mask_repeats']})
display(pd.DataFrame(comparison).T[['Recall@10','Recall@20','Recall@50','NDCG@20']])
generalization={f'{model_label} {group}':test_evaluation[no_prior][group] for group in ('seen','unseen')}
generalization.update({f'Hybrid {group}':test_evaluation[best_hybrid_config][group] for group in ('seen','unseen')})
display(pd.DataFrame(generalization).T[['tasks','Recall@10','Recall@20','Recall@50','NDCG@10','NDCG@20']])

ablation_rows=[]
if CFG['run_ablations']:
    model.to('cpu'); torch.cuda.empty_cache()
    for label,variational_ablation,freeze_ablation in CFG['ablations']:
        checkpoint=ROOT/f'checkpoints/attention_{IDENTITY_SCHEMA}_{label}_{CFG["card2vec_dim"]}.pt'
        ablation_model=build_model(variational_ablation,freeze_ablation)
        ablation_model,ablation_history,ablation_best=train_experiment(ablation_model,checkpoint,variational_ablation,label)
        validation=ablation_best['val_metrics']; test=evaluate_vae_batched(test_decks,(no_prior,),CFG['primary_mask_ratio'],CFG['test_mask_repeats'],candidate_model=ablation_model)[no_prior]['overall']
        ablation_rows.append({'experiment':label,'best_epoch':ablation_best['epoch'],**{f'validation_{key}':value for key,value in validation.items() if key!='tasks'},**{f'test_{key}':value for key,value in test.items() if key!='tasks'}})
        ablation_model.to('cpu'); del ablation_model,ablation_history; gc.collect(); torch.cuda.empty_cache()
    model.to(device)
    display(pd.DataFrame(ablation_rows).set_index('experiment'))
else:
    print('Ablations disabled; set RUN_ABLATIONS=True in the configuration cell to run them.')


## 10 — Manual EDHREC versus model comparisons

The three case cells below contain normal Python data, not copied webpage text: a Commander name, a card-to-quantity mapping for the known deck, and ordered card-name lists from EDHREC and the model. List position is recommendation order, so no explicit rank, price, inclusion count, model score, CSV, or text parser is needed.

Every name is resolved directly to its canonical Oracle ID before comparison, and cards already present in the partial deck are removed. This remains an **engine-agreement analysis**, not a ground-truth accuracy test: EDHREC is another recommender, not the correct completion.

Run cells 20–25 for combined statistics. Cells 26–28 show each Commander in detail, and cell 29 plots aggregate profiles. To add a case, copy a case cell, paste clean card names into its structures, and append the dictionary to `COMPARISON_CASES`.


In [ ]:
from pathlib import Path
from collections import Counter
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from mtgdeck.legality import OracleCatalog

if 'catalog' not in globals():
    snapshots=sorted((ROOT/'data').glob('oracle_cards_*.jsonl.gz'),key=lambda path:path.stat().st_mtime)
    oracle_path=snapshots[-1] if snapshots else ROOT/'data/oracle_cards.json'
    catalog=OracleCatalog.from_path(oracle_path,ROOT/'data/commander_eligible_oracle_ids.json')

def resolve_card(name):
    card=catalog.resolve(name)
    if card is None: raise ValueError(f'Card was not found in the Oracle snapshot: {name}')
    return card

def oracle_row(card,**extra):
    type_line=str(card.get('type_line',''))
    if 'Creature' in type_line: category='Creature'
    elif 'Land' in type_line: category='Land'
    elif 'Artifact' in type_line: category='Artifact'
    elif 'Enchantment' in type_line: category='Enchantment'
    elif 'Planeswalker' in type_line: category='Planeswalker'
    else: category='Instant/Sorcery'
    return {'oracle_id':str(card['oracle_id']),'card':str(card['name']),'type_line':type_line,'category':category,'cmc':float(card.get('cmc') or 0),**extra}

def resolve_ranked_names(names):
    rows=[]; seen=set()
    for name in names:
        card=resolve_card(name); oracle_id=str(card['oracle_id'])
        if oracle_id in seen: raise ValueError(f'Duplicate recommendation: {card["name"]}')
        seen.add(oracle_id); rows.append(oracle_row(card))
    return rows

def normalize_case(case):
    commander=resolve_card(case['commander'])
    partial=[oracle_row(resolve_card(name),quantity=int(quantity)) for name,quantity in case['partial_cards'].items()]
    edhrec=resolve_ranked_names(case['edhrec_cards']); model_rows=resolve_ranked_names(case['model_cards'])
    visible={str(commander['oracle_id']),*(row['oracle_id'] for row in partial)}
    def filter_visible(rows):
        filtered=[dict(row) for row in rows if row['oracle_id'] not in visible]
        for rank,row in enumerate(filtered,1): row['rank']=rank
        return filtered
    return {**case,'commander_card':commander,'partial':partial,'edhrec':filter_visible(edhrec),'model':filter_visible(model_rows)}

def rank_biased_overlap(left,right,k,p=0.9):
    left=[row['oracle_id'] for row in left[:k]]; right=[row['oracle_id'] for row in right[:k]]
    depth=min(k,len(left),len(right))
    if not depth: return 0.0
    seen_left=set(); seen_right=set(); weighted=0.0; overlap=0
    for index in range(depth):
        seen_left.add(left[index]); seen_right.add(right[index]); overlap=len(seen_left&seen_right)
        weighted+=(1-p)*(p**index)*(overlap/(index+1))
    return weighted+(p**depth)*(overlap/depth)

def agreement_row(case,k):
    model_top=case['model'][:k]; edhrec_top=case['edhrec'][:k]
    model_ids={row['oracle_id'] for row in model_top}; edhrec_ids={row['oracle_id'] for row in edhrec_top}; common=model_ids&edhrec_ids; union=model_ids|edhrec_ids
    model_ranks={row['oracle_id']:row['rank'] for row in model_top}; edhrec_ranks={row['oracle_id']:row['rank'] for row in edhrec_top}
    spearman=float('nan')
    if len(common)>=2:
        shared=sorted(common); spearman=pd.Series([model_ranks[x] for x in shared]).corr(pd.Series([edhrec_ranks[x] for x in shared]),method='spearman')
    return {'commander':case['commander_card']['name'],'K':k,'Common@K':len(common),'Agreement@K':len(common)/max(1,min(len(model_top),len(edhrec_top))),'Jaccard@K':len(common)/max(1,len(union)),'RBO@K':rank_biased_overlap(model_top,edhrec_top,k),'Shared rank Spearman':spearman}

def recommendation_profile(case,engine,k=25):
    rows=case[engine][:k]; counts=Counter(row['category'] for row in rows)
    result={'commander':case['commander_card']['name'],'engine':'Model' if engine=='model' else 'EDHREC','K':min(k,len(rows)),'Mean mana value':np.mean([row['cmc'] for row in rows]) if rows else np.nan}
    for category in ('Land','Creature','Artifact','Enchantment','Instant/Sorcery','Planeswalker'):
        result[f'{category} share']=counts[category]/max(1,len(rows))
    return result

def show_case(case_key,k=25):
    case=NORMALIZED_CASES[case_key]; model_rows=case['model'][:k]; edhrec_rows=case['edhrec'][:k]; width=max(len(model_rows),len(edhrec_rows))
    side=[{'Rank':index+1,'Model card':model_rows[index]['card'] if index<len(model_rows) else None,'EDHREC card':edhrec_rows[index]['card'] if index<len(edhrec_rows) else None} for index in range(width)]
    model_rank={row['oracle_id']:row['rank'] for row in model_rows}; edhrec_rank={row['oracle_id']:row['rank'] for row in edhrec_rows}; common=set(model_rank)&set(edhrec_rank)
    name_by_id={row['oracle_id']:row['card'] for row in [*model_rows,*edhrec_rows]}
    shared=pd.DataFrame([{'Card':name_by_id[oracle_id],'Model rank':model_rank[oracle_id],'EDHREC rank':edhrec_rank[oracle_id],'Absolute rank difference':abs(model_rank[oracle_id]-edhrec_rank[oracle_id])} for oracle_id in common]).sort_values(['Model rank','EDHREC rank']) if common else pd.DataFrame(columns=['Card','Model rank','EDHREC rank','Absolute rank difference'])
    print(case['commander_card']['name'],f'— top {k}'); display(pd.DataFrame(side)); print('Shared recommendations'); display(shared)


In [ ]:
CHULANE_CASE = {
    "key": "chulane",
    "commander": "Chulane, Teller of Tales",
    "partial_cards": {
        "Arcane Signet": 1,
        "Cloudstone Curio": 1,
        "Lightning Greaves": 1,
        "Sol Ring": 1,
        "Thought Vessel": 1,
        "Azorius Chancery": 1,
        "Breeding Pool": 1,
        "Canopy Vista": 1,
        "Command Tower": 1,
        "Exotic Orchard": 1,
        "Flooded Strand": 1,
        "Forest": 5,
        "Glacial Fortress": 1,
        "Hallowed Fountain": 1,
        "Hinterland Harbor": 1,
        "Island": 5,
        "Misty Rainforest": 1,
        "Plains": 4,
        "Prairie Stream": 1,
        "Reliquary Tower": 1,
        "Rejuvenating Springs": 1,
        "Seaside Citadel": 1,
        "Selesnya Sanctuary": 1,
        "Simic Growth Chamber": 1,
        "Spara's Headquarters": 1,
        "Sunpetal Grove": 1,
        "Temple Garden": 1,
        "Windswept Heath": 1,
        "Arbor Elf": 1,
        "Avacyn's Pilgrim": 1,
        "Avenger of Zendikar": 1,
        "Birds of Paradise": 1,
        "Beast Whisperer": 1,
        "Bloom Tender": 1,
        "Coiling Oracle": 1,
        "Craterhoof Behemoth": 1,
        "Delighted Halfling": 1,
        "Destiny Spinner": 1,
        "Dour Port-Mage": 1,
        "Dream Stalker": 1,
        "Elvish Mystic": 1,
        "Esper Sentinel": 1,
        "Eternal Witness": 1,
        "Faeburrow Elder": 1,
        "Frilled Mystic": 1,
        "Fyndhorn Elves": 1,
        "Grand Abolisher": 1,
        "Laboratory Maniac": 1,
        "Llanowar Elves": 1,
        "Lotus Cobra": 1,
        "Mulldrifter": 1,
        "Mystic Snake": 1,
        "Noble Hierarch": 1,
        "Paradise Druid": 1,
        "Peregrine Drake": 1,
        "Rampaging Baloths": 1,
        "Reclamation Sage": 1,
        "Sakura-Tribe Elder": 1,
        "Scute Swarm": 1,
        "Seedborn Muse": 1,
        "Shrieking Drake": 1,
        "Tatyova, Benthic Druid": 1,
        "Tireless Provisioner": 1,
        "Whitemane Lion": 1,
        "Wood Elves": 1,
        "Aluren": 1,
        "Guardian Project": 1,
        "Intruder Alarm": 1,
        "Mana Breach": 1,
        "Mystic Remora": 1,
        "Rhystic Study": 1,
        "An Offer You Can't Refuse": 1,
        "Beast Within": 1,
        "Counterspell": 1,
        "Cyclonic Rift": 1,
        "Dovin's Veto": 1,
        "Eladamri's Call": 1,
        "Heroic Intervention": 1,
        "Path to Exile": 1,
        "Swords to Plowshares": 1,
        "Worldly Tutor": 1,
        "Cultivate": 1,
        "Farseek": 1,
        "Finale of Devastation": 1,
        "Kodama's Reach": 1,
        "Rampant Growth": 1
    },
    "edhrec_cards": [
        "Bountiful Promenade",
        "Sea of Clouds",
        "Yavimaya Coast",
        "Boseiju, Who Endures",
        "Swan Song",
        "Enlightened Tutor",
        "Dreamroot Cascade",
        "Fierce Guardianship",
        "Otawara, Soaring City",
        "Yavimaya, Cradle of Growth",
        "Nature's Lore",
        "Brushland",
        "Adarkar Wastes",
        "Tropical Island",
        "Savannah",
        "Tundra",
        "Overgrown Farmland",
        "Evolving Wilds",
        "Generous Gift",
        "Chord of Calling",
        "Three Visits",
        "Negate",
        "Teferi's Protection",
        "Swiftfoot Boots",
        "Arcane Denial",
        "Kinnan, Bonder Prodigy",
        "City of Brass",
        "Inga and Esika",
        "Deserted Beach",
        "Mana Drain",
        "Felidar Retreat",
        "Aura Shards",
        "Growth Spiral",
        "Mana Confluence",
        "Ranger-Captain of Eos",
        "Enduring Vitality",
        "Fabled Passage",
        "Aven Mindcensor",
        "Smothering Tithe",
        "Wooded Foothills",
        "Gaea's Cradle",
        "Silence",
        "Panharmonicon",
        "Terramorphic Expanse",
        "Drannith Magistrate",
        "Flooded Grove",
        "Verdant Catacombs",
        "Polluted Delta",
        "Force of Will",
        "Temple of Mystery",
        "Incubation Druid",
        "Priest of Titania",
        "Scalding Tarn",
        "Ancient Tomb",
        "Badgermole Cub",
        "Kutzil, Malamet Exemplar",
        "Recruiter of the Guard",
        "Sun Titan",
        "Crop Rotation",
        "Wargate",
        "Alchemist's Refuge",
        "Collector Ouphe",
        "Path of Ancestry",
        "Eldritch Evolution",
        "Karametra, God of Harvests",
        "Thassa's Oracle",
        "Farewell",
        "Veil of Summer",
        "Tranquil Landscape",
        "Retreat to Coralhelm",
        "Aesi, Tyrant of Gyre Strait",
        "Neoform",
        "Force of Negation",
        "Flusterstorm",
        "Temple of Plenty",
        "Green Sun's Zenith",
        "Temple of Enlightenment",
        "Sylvan Library",
        "Delney, Streetwise Lookout",
        "Marsh Flats",
        "Kor Skyfisher",
        "Lavinia, Azorius Renegade",
        "Overburden",
        "Arid Mesa",
        "Village Bell-Ringer",
        "Skyclave Apparition",
        "Elvish Visionary",
        "Hedge Maze",
        "Voice of Victory",
        "The Great Henge",
        "Temple of the False God",
        "Farhaven Elf",
        "Cloud of Faeries",
        "Nissa, Resurgent Animist",
        "Deafening Silence",
        "Austere Command",
        "Supreme Verdict",
        "Hullbreaker Horror",
        "Gemstone Caverns",
        "Lush Portico"
    ],
    "model_cards": [
        "Nature's Lore",
        "Yavimaya Coast",
        "Swan Song",
        "Yavimaya, Cradle of Growth",
        "Three Visits",
        "Generous Gift",
        "Arcane Denial",
        "Sea of Clouds",
        "Growth Spiral",
        "Dreamroot Cascade",
        "Negate",
        "Bountiful Promenade",
        "Cloud of Faeries",
        "Deadeye Navigator",
        "Smothering Tithe",
        "Supreme Verdict",
        "Chord of Calling",
        "Aura Shards",
        "Swiftfoot Boots",
        "Adarkar Wastes",
        "Alchemist's Refuge",
        "Fabled Passage",
        "Pongify",
        "Wooded Foothills",
        "Fierce Guardianship"
    ],
    "model_label": "Variational fine-tuned 896-d checkpoint"
}
print(CHULANE_CASE['commander'], {'partial unique':len(CHULANE_CASE['partial_cards']),'EDHREC recommendations':len(CHULANE_CASE['edhrec_cards']),'model recommendations':len(CHULANE_CASE['model_cards'])})


In [ ]:
UR_DRAGON_CASE = {
    "key": "ur_dragon",
    "commander": "The Ur-Dragon",
    "partial_cards": {
        "Arcane Signet": 1,
        "Chromatic Lantern": 1,
        "Dragon's Hoard": 1,
        "Fellwar Stone": 1,
        "Herald's Horn": 1,
        "Lightning Greaves": 1,
        "Mox Jasper": 1,
        "Sol Ring": 1,
        "Urza's Incubator": 1,
        "Arid Mesa": 1,
        "Blood Crypt": 1,
        "Bloodstained Mire": 1,
        "Breeding Pool": 1,
        "Cavern of Souls": 1,
        "Command Tower": 1,
        "Exotic Orchard": 1,
        "Flooded Strand": 1,
        "Forest": 2,
        "Haven of the Spirit Dragon": 1,
        "Island": 2,
        "Ketria Triome": 1,
        "Maelstrom of the Spirit Dragon": 1,
        "Misty Rainforest": 1,
        "Mountain": 2,
        "Overgrown Tomb": 1,
        "Path of Ancestry": 1,
        "Plains": 1,
        "Polluted Delta": 1,
        "Reflecting Pool": 1,
        "Sacred Foundry": 1,
        "Scalding Tarn": 1,
        "Steam Vents": 1,
        "Stomping Ground": 1,
        "Swamp": 2,
        "Temple Garden": 1,
        "The World Tree": 1,
        "Verdant Catacombs": 1,
        "Watery Grave": 1,
        "Windswept Heath": 1,
        "Wooded Foothills": 1,
        "Ancient Copper Dragon": 1,
        "Ancient Gold Dragon": 1,
        "Ancient Silver Dragon": 1,
        "Atarka, World Render": 1,
        "Birds of Paradise": 1,
        "Balefire Dragon": 1,
        "Bladewing the Risen": 1,
        "Dragonlord Dromoka": 1,
        "Dragonlord's Servant": 1,
        "Dragonspeaker Shaman": 1,
        "Goldspan Dragon": 1,
        "Hellkite Courser": 1,
        "Klauth, Unrivaled Ancient": 1,
        "Lathliss, Dragon Queen": 1,
        "Miirym, Sentinel Wyrm": 1,
        "Morophon, the Boundless": 1,
        "Old Gnawbone": 1,
        "Ramos, Dragon Engine": 1,
        "Rith, Liberated Primeval": 1,
        "Rivaz of the Claw": 1,
        "Roaming Throne": 1,
        "Sarkhan, Soul Aflame": 1,
        "Savage Ventmaw": 1,
        "Scion of Draco": 1,
        "Scion of the Ur-Dragon": 1,
        "Scourge of Valkas": 1,
        "Silumgar, the Drifting Death": 1,
        "Terror of the Peaks": 1,
        "Tiamat": 1,
        "Twinflame Tyrant": 1,
        "Ureni of the Unwritten": 1,
        "Utvara Hellkite": 1,
        "Call the Spirit Dragons": 1,
        "Dracogenesis": 1,
        "Dragon Tempest": 1,
        "Garruk's Uprising": 1,
        "Rhystic Study": 1,
        "Rhythm of the Wild": 1
    },
    "edhrec_cards": [
        "Crux of Fate",
        "Temur Ascendancy",
        "Farseek",
        "Nature's Lore",
        "Three Visits",
        "Swords to Plowshares",
        "Heroic Intervention",
        "Hallowed Fountain",
        "Marsh Flats",
        "Godless Shrine",
        "Smothering Tithe",
        "Cyclonic Rift",
        "Bloom Tender",
        "Cultivate",
        "Teferi's Protection",
        "Two-Headed Hellkite",
        "Ziatora's Proving Ground",
        "Ancient Brass Dragon",
        "Scalelord Reckoner",
        "Jetmir's Garden",
        "City of Brass",
        "Demonic Tutor",
        "Zagoth Triome",
        "Mana Confluence",
        "Kodama's Reach",
        "Path to Exile",
        "Savai Triome",
        "Unclaimed Territory",
        "The Great Henge",
        "Indatha Triome",
        "Secluded Courtyard",
        "Sarkhan Unbroken",
        "Ganax, Astral Hunter",
        "Raugrin Triome",
        "Hellkite Tyrant",
        "Counterspell",
        "Mana Drain",
        "Kindred Discovery",
        "Skyshroud Claim",
        "Thrakkus the Butcher",
        "Korlessa, Scale Singer",
        "Orb of Dragonkind",
        "Sarkhan's Triumph",
        "Assassin's Trophy",
        "Hellkite Charger",
        "Reliquary Tower",
        "Crucible of the Spirit Dragon",
        "Wrathful Red Dragon",
        "Rampant Growth",
        "Temple of the Dragon Queen",
        "Carnelian Orb of Dragonkind",
        "Spara's Headquarters",
        "Smaug, Wicked Worm",
        "Dragonstorm Globe",
        "Up the Beanstalk",
        "Taiga",
        "Vampiric Tutor",
        "Zurgo and Ojutai",
        "Scourge of the Throne",
        "Betor, Kin to All",
        "Scavenger Regent // Exude Toxin",
        "Volcanic Island",
        "Tropical Island",
        "Bayou",
        "Badlands",
        "Xander's Lounge",
        "Savannah",
        "Dragonlord Kolaghan",
        "Plateau",
        "Delighted Halfling",
        "Worldly Tutor",
        "Ancient Bronze Dragon",
        "Scaled Nurturer",
        "Frontier Bivouac",
        "Fist of Suns",
        "Karrthus, Tyrant of Jund",
        "Swan Song",
        "Ignoble Hierarch",
        "Underground Sea",
        "Mirari's Wake",
        "Tundra",
        "Stubborn Denial",
        "Boseiju, Who Endures",
        "Scrubland",
        "Ancient Tomb",
        "Crucible of Fire",
        "Commander's Sphere",
        "The One Ring",
        "Dragonlord Silumgar",
        "Elemental Bond",
        "Broodcaller Scourge",
        "Majestic Genesis",
        "Cavern-Hoard Dragon",
        "Teneb, the Harvester",
        "Three Tree City",
        "Anguished Unmaking",
        "Jungle Shrine",
        "Spire Garden",
        "Kolaghan, the Storm's Fury",
        "Ojutai, Soul of Winter"
    ],
    "model_cards": [
        "Crux of Fate",
        "Farseek",
        "Nature's Lore",
        "Temur Ascendancy",
        "Three Visits",
        "Hallowed Fountain",
        "Two-Headed Hellkite",
        "City of Brass",
        "Heroic Intervention",
        "Bloom Tender",
        "Mana Confluence",
        "Swords to Plowshares",
        "Cultivate",
        "Ancient Brass Dragon",
        "Godless Shrine",
        "Kodama's Reach",
        "Teferi's Protection",
        "Marsh Flats",
        "Scalelord Reckoner",
        "Savai Triome",
        "Cyclonic Rift",
        "Jetmir's Garden",
        "Korlessa, Scale Singer",
        "Ganax, Astral Hunter",
        "Hellkite Tyrant"
    ],
    "model_label": "Variational fine-tuned 896-d checkpoint"
}
print(UR_DRAGON_CASE['commander'], {'partial unique':len(UR_DRAGON_CASE['partial_cards']),'EDHREC recommendations':len(UR_DRAGON_CASE['edhrec_cards']),'model recommendations':len(UR_DRAGON_CASE['model_cards'])})


In [ ]:
TSABO_TAVOC_CASE = {
    "key": "tsabo_tavoc",
    "commander": "Tsabo Tavoc",
    "partial_cards": {
        "Arcane Signet": 1,
        "Charcoal Diamond": 1,
        "Commander's Sphere": 1,
        "Fellwar Stone": 1,
        "Lightning Greaves": 1,
        "Mind Stone": 1,
        "Rakdos Signet": 1,
        "Sol Ring": 1,
        "Swiftfoot Boots": 1,
        "Talisman of Indulgence": 1,
        "Thran Dynamo": 1,
        "Tsabo's Web": 1,
        "Blood Crypt": 1,
        "Bloodstained Mire": 1,
        "Bojuka Bog": 1,
        "Command Tower": 1,
        "Dragonskull Summit": 1,
        "Luxury Suite": 1,
        "Black Market": 1,
        "Havoc Festival": 1,
        "No Mercy": 1,
        "Painful Quandary": 1,
        "Phyrexian Arena": 1,
        "Phyrexian Reclamation": 1,
        "Abrade": 1,
        "Backlash": 1,
        "Bedevil": 1,
        "Chaos Warp": 1,
        "Dark Ritual": 1,
        "Deflecting Swat": 1,
        "Rakdos Charm": 1,
        "Terminate": 1,
        "Tsabo's Decree": 1,
        "Withering Torment": 1,
        "Blasphemous Act": 1,
        "Damnation": 1,
        "Dreadbore": 1,
        "Faithless Looting": 1,
        "Feed the Swarm": 1,
        "Night's Whisper": 1,
        "Read the Bones": 1,
        "Reanimate": 1,
        "Sign in Blood": 1,
        "Vandalblast": 1
    },
    "edhrec_cards": [
        "Swamp",
        "Mountain",
        "Smoldering Marsh",
        "Sulfurous Springs",
        "Tainted Peak",
        "Rakdos Carnarium",
        "Urborg, Tomb of Yawgmoth",
        "Solemn Simulacrum",
        "Mogis, God of Slaughter",
        "Kaervek the Merciless",
        "Temple of Malice",
        "Foreboding Ruins",
        "Royal Assassin",
        "Kardur, Doomscourge",
        "Avatar of Woe",
        "The Lord of Pain",
        "Anger",
        "Master of Cruelties",
        "Tsabo's Assassin",
        "Shadowblood Ridge",
        "Haunted Ridge",
        "Harsh Mentor",
        "Dread",
        "Burnished Hart",
        "Blazemire Verge",
        "Archfiend of Depravity",
        "Sheoldred, Whispering One",
        "Graven Cairns",
        "Canyon Slough",
        "Jeska's Will",
        "Cabal Coffers",
        "Orcish Bowmasters",
        "Urabrask the Hidden",
        "Jet Medallion",
        "Lagomos, Hand of Hatred",
        "Exotic Orchard",
        "Wayfarer's Bauble",
        "Demonic Tutor",
        "Phyrexian Delver",
        "Rogue's Passage",
        "Rakdos, Lord of Riots",
        "Viashino Heretic",
        "Animate Dead",
        "Black Market Connections",
        "Patriar's Seal",
        "Gleeful Arsonist",
        "Massacre Wurm",
        "Gray Merchant of Asphodel",
        "Raucous Theater",
        "Seething Song",
        "Captive Audience",
        "Reliquary Tower",
        "Temple of the False God",
        "Blightstep Pathway // Searstep Pathway",
        "Blood Artist",
        "Angrath, the Flame-Chained",
        "Evolving Wilds",
        "Myriad Landscape",
        "Lightning Bolt",
        "Fire Diamond",
        "Deadly Rollick",
        "Visara the Dreadful",
        "Ruby Medallion",
        "Deadly Dispute",
        "Toxic Deluge",
        "Victimize",
        "Etali, Primal Storm",
        "Shakedown Heavy",
        "Go for the Throat",
        "Angrath's Rampage",
        "Rising of the Day",
        "Vampiric Tutor",
        "Badlands",
        "Tibalt's Trickery",
        "Thought Vessel",
        "Blackcleave Cliffs",
        "Bloodfell Caves",
        "Morbid Opportunist",
        "Terramorphic Expanse",
        "Thousand-Year Elixir",
        "Gilded Lotus",
        "Dauthi Voidwalker",
        "Thrill of Possibility",
        "Big Score",
        "Fell the Profane // Fell Mire",
        "Ancient Tomb",
        "Diabolic Tutor",
        "Exsanguinate",
        "Mount Doom",
        "Malakir Rebirth // Malakir Mire",
        "Decree of Pain",
        "Phyrexian Purge",
        "Whip of Erebos",
        "Necropotence",
        "Razorkin Needlehead",
        "Cabal Ritual",
        "Sheoldred, the Apocalypse",
        "Imp's Mischief",
        "Phyrexian Tower",
        "Fabled Passage"
    ],
    "model_cards": [
        "Mountain",
        "Swamp",
        "Smoldering Marsh",
        "Haunted Ridge",
        "Sulfurous Springs",
        "Foreboding Ruins",
        "Reliquary Tower",
        "Toxic Deluge",
        "Thought Vessel",
        "Blightstep Pathway // Searstep Pathway",
        "Jeska's Will",
        "Tainted Peak",
        "Demonic Tutor",
        "Heartless Hidetsugu",
        "Temple of Malice",
        "Gray Merchant of Asphodel",
        "Graven Cairns",
        "Exotic Orchard",
        "Rakdos Carnarium",
        "Wayfarer's Bauble",
        "Deadly Rollick",
        "Torment of Hailfire",
        "Solemn Simulacrum",
        "Shadowblood Ridge",
        "Cabal Coffers"
    ],
    "model_label": "Variational fine-tuned 896-d checkpoint"
}
print(TSABO_TAVOC_CASE['commander'], {'partial unique':len(TSABO_TAVOC_CASE['partial_cards']),'EDHREC recommendations':len(TSABO_TAVOC_CASE['edhrec_cards']),'model recommendations':len(TSABO_TAVOC_CASE['model_cards'])})


In [ ]:
COMPARISON_CASES=[CHULANE_CASE,UR_DRAGON_CASE,TSABO_TAVOC_CASE]
NORMALIZED_CASES={case['key']:normalize_case(case) for case in COMPARISON_CASES}
normalization_audit=[]
for case in NORMALIZED_CASES.values():
    physical=sum(row['quantity'] for row in case['partial']); known_cards=physical+1
    normalization_audit.append({'commander':case['commander_card']['name'],'partial physical cards':physical,'partial unique cards':len(case['partial']),'known cards incl. commander':known_cards,'nominal open slots':max(0,100-known_cards),'EDHREC recommendations':len(case['edhrec']),'model recommendations':len(case['model'])})
normalization_audit=pd.DataFrame(normalization_audit).set_index('commander')
display(normalization_audit)


In [ ]:
AGREEMENT_DEPTHS=(5,10,20,25)
agreement_stats=pd.DataFrame([agreement_row(case,k) for case in NORMALIZED_CASES.values() for k in AGREEMENT_DEPTHS])
display(agreement_stats.set_index(['commander','K']).round(3))
print('Macro-average across the three commanders')
display(agreement_stats.groupby('K')[['Common@K','Agreement@K','Jaccard@K','RBO@K']].mean().round(3))

recommendation_profiles=pd.DataFrame([recommendation_profile(case,engine,25) for case in NORMALIZED_CASES.values() for engine in ('model','edhrec')])
display(recommendation_profiles.set_index(['commander','engine']).round(3))


In [ ]:
show_case('chulane',25)


In [ ]:
show_case('ur_dragon',25)


In [ ]:
show_case('tsabo_tavoc',25)


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for commander,rows in agreement_stats.groupby('commander'):
    axes[0].plot(rows['K'],rows['Agreement@K'],marker='o',label=commander)
axes[0].set(title='Top-K recommendation agreement',xlabel='K',ylabel='Common cards / K',ylim=(0,1)); axes[0].grid(alpha=.25); axes[0].legend()

profile_plot=recommendation_profiles.set_index(['commander','engine'])[[column for column in recommendation_profiles if column.endswith(' share')]]
profile_plot.plot(kind='bar',stacked=True,ax=axes[1],colormap='tab20c')
axes[1].set(title='Top-25 recommendation mix',xlabel='',ylabel='Share',ylim=(0,1)); axes[1].legend(loc='center left',bbox_to_anchor=(1,0.5),fontsize=8)
plt.tight_layout(); plt.show()

# Optional exports after inspection:
# agreement_stats.to_csv(ROOT/'data/manual_engine_agreement.csv',index=False)
# recommendation_profiles.to_csv(ROOT/'data/manual_engine_profiles.csv',index=False)
